# Costruire il Primo Agente AI con `smolagents`

In questo notebook costruiremo un agente AI passo dopo passo.

Un **agente AI** è un sistema che:
1. Riceve un obiettivo in linguaggio naturale
2. **Ragiona** su come raggiungerlo (Think)
3. Sceglie e **esegue azioni** tramite tool (Act)
4. **Osserva** il risultato (Observe)
5. Ripete il ciclo finché non raggiunge la risposta finale

```
┌─────────────────────────────────────────┐
│            CICLO DELL'AGENTE            │
│                                         │
│   Obiettivo → THINK → ACT → OBSERVE     │
│                  ↑_____________│        |
└─────────────────────────────────────────┘
```

Useremo **`smolagents`**, la libreria di HuggingFace per costruire agenti in modo semplice.

## 1. Installazione delle Dipendenze

Se stai usando l'ambiente `uv` di questo progetto, le dipendenze sono già installate.

Se invece vuoi installare in un ambiente Colab o diverso, esegui la cella qui sotto (rimuovi il `#`):

In [ ]:
# Decommentare solo se NON si usa l'ambiente uv del progetto
# %pip install smolagents gradio pytz requests pyyaml python-dotenv duckduckgo-search huggingface_hub

## 2. Import delle Librerie

Importiamo tutto quello che ci serve:
- `smolagents`: il framework per gli agenti
- `dotenv`: per caricare il token HF da un file `.env` senza mai esporlo nel codice
- `datetime`, `pytz`: per lavorare con date e fusi orari

In [10]:
import os
import datetime
import pytz

from dotenv import load_dotenv
from smolagents import (
    CodeAgent,           # Il tipo di agente che scrive ed esegue codice Python
    InferenceClientModel, # Il modello LLM servito via API HuggingFace
    DuckDuckGoSearchTool, # Tool built-in per cercare su internet
    tool,                # Decoratore per trasformare funzioni in tool
)

print("Librerie importate correttamente!")

Librerie importate correttamente!


## 3. Autenticazione HuggingFace

Per usare i modelli via API di HuggingFace è necessario un token.

**Come ottenere il token:**
1. Vai su https://hf.co/settings/tokens
2. Crea un nuovo token con permessi `write`
3. Copia il file `.env.example` in `.env` nella cartella del progetto
4. Inserisci il tuo token: `HF_TOKEN=hf_tuoTokenQui`

> **Sicurezza:** il file `.env` è elencato nel `.gitignore`, quindi non verrà mai committato su Git. Non condividere mai il tuo token!

In [11]:
# Carica le variabili d'ambiente dal file .env
load_dotenv()

hf_token = os.getenv("HF_TOKEN")

if hf_token:
    # Mostriamo solo i primi 8 caratteri per conferma, mai il token completo
    print(f"Token HF caricato correttamente: {hf_token[:8]}...")
else:
    print("ATTENZIONE: HF_TOKEN non trovato!")
    print("Crea un file .env con: HF_TOKEN=hf_tuoTokenQui")

Token HF caricato correttamente: hf_GxbIC...


## 4. I Tool: Le "Mani" dell'Agente

I **tool** sono funzioni Python che l'agente può chiamare per interagire con il mondo esterno.

Per creare un tool valido con `smolagents` bisogna:
1. Decorare la funzione con `@tool`
2. **Specificare i tipi** degli argomenti e del valore di ritorno (type hints)
3. Scrivere un **docstring chiaro** con la descrizione della funzione e di ogni argomento

Il docstring è fondamentale: l'LLM lo legge per capire *quando* e *come* usare il tool.

In [12]:
# ----- TOOL 1: Ora locale in un fuso orario -----

@tool
def get_current_time_in_timezone(timezone: str) -> str:
    """Restituisce l'ora locale corrente in un fuso orario specificato.
    Args:
        timezone: Una stringa che rappresenta un fuso orario valido (es. 'Europe/Rome', 'America/New_York').
    """
    try:
        tz = pytz.timezone(timezone)
        local_time = datetime.datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
        return f"L'ora locale in {timezone} è: {local_time}"
    except Exception as e:
        return f"Errore per il fuso orario '{timezone}': {str(e)}"


# Proviamo il tool direttamente (senza agente) per verificare che funzioni
print(get_current_time_in_timezone("Europe/Rome"))
print(get_current_time_in_timezone("America/New_York"))

L'ora locale in Europe/Rome è: 2026-05-12 10:56:38
L'ora locale in America/New_York è: 2026-05-12 04:56:38


In [13]:
# ----- TOOL 2: Calcolatrice semplice -----

@tool
def calculate(expression: str) -> str:
    """Calcola il risultato di un'espressione matematica testuale.
    Args:
        expression: L'espressione matematica da calcolare (es. '2 + 2', '15 * 4', '100 / 3').
    """
    try:
        # eval è sicuro qui perché limitiamo le operazioni matematiche
        allowed = set("0123456789+-*/()., ")
        if not all(c in allowed for c in expression):
            return "Errore: solo operazioni matematiche base sono permesse."
        result = eval(expression)
        return f"Risultato di '{expression}' = {result}"
    except Exception as e:
        return f"Errore nel calcolo: {str(e)}"


# Test del tool
print(calculate("(15 * 4) + 20"))
print(calculate("100 / 3"))

Risultato di '(15 * 4) + 20' = 80
Risultato di '100 / 3' = 33.333333333333336


## 5. Tool Built-in di smolagents

`smolagents` include già dei tool pronti all'uso. Il più utile è `DuckDuckGoSearchTool`, che permette all'agente di cercare informazioni su Internet.

Vediamo la lista dei tool che useremo:
| Tool | Descrizione |
|------|-------------|
| `DuckDuckGoSearchTool` | Cerca sul web via DuckDuckGo |
| `get_current_time_in_timezone` | Ora in un fuso orario (custom) |
| `calculate` | Calcolatrice (custom) |

In [14]:
# Istanziamo il tool di ricerca web built-in
search_tool = DuckDuckGoSearchTool()

# Testiamo anche questo direttamente
result = search_tool("HuggingFace smolagents framework")
print(result[:500], "...")  # Mostriamo solo i primi 500 caratteri

## Search Results

[Smolagents : Huggingface AI Agent Framework](https://smolagents.org/)
Smolagents is a minimalist AI agent framework developed by the Hugging Face team, crafted to enable developers to deploy robust agents with just a ...

[huggingface/smolagents | DeepWiki](https://deepwiki.com/huggingface/smolagents)
CodeAgent path: The LLM output is parsed by parse_code_blobs ( src/smolagents/utils.py ), then executed by a PythonExecutor .

[Getting Started | huggingface/smolagents | DeepWi ...


## 6. Il Modello LLM

L'**LLM** è il "cervello" dell'agente: riceve il contesto (obiettivo + risultati dei tool) e decide cosa fare.

Usiamo `InferenceClientModel` di smolagents, che si connette all'API serverless di HuggingFace.

Il modello scelto è **`Qwen/Qwen2.5-Coder-32B-Instruct`**: ottimo per ragionare e scrivere codice.

In [15]:
# Creiamo il modello LLM che userà l'API di HuggingFace
model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    token=hf_token,          # Il token che abbiamo caricato dal .env
    max_tokens=2048,
    temperature=0.5,          # 0 = deterministico, 1 = creativo
)

print(f"Modello pronto: {model.model_id}")

Modello pronto: Qwen/Qwen2.5-Coder-32B-Instruct


## 7. Creare l'Agente

Ora assembliamo tutto in un `CodeAgent`.

**Cos'è un CodeAgent?**  
È un tipo di agente che esprime le sue azioni scrivendo **blocchi di codice Python**. Invece di chiamare i tool come JSON, il modello scrive codice che viene eseguito nel runtime. Questo lo rende molto potente per task computazionali.

**Parametri chiave:**
- `model`: l'LLM da usare
- `tools`: lista di tool disponibili
- `max_steps`: numero massimo di passi Think→Act→Observe prima di fermarsi

In [16]:
# Raccogliamo tutti i tool in una lista
tools = [
    search_tool,                    # Ricerca web
    get_current_time_in_timezone,   # Ora nel mondo
    calculate,                      # Calcolatrice
]

# Creiamo il CodeAgent
agent = CodeAgent(
    model=model,
    tools=tools,
    max_steps=5,        # Max 5 cicli Think→Act→Observe
    verbosity_level=2,  # 0=silenzioso, 1=normale, 2=dettagliato
)

print("Agente creato!")
print(f"Tool disponibili: {[t.name for t in agent.tools.values()]}")

Agente creato!
Tool disponibili: ['web_search', 'get_current_time_in_timezone', 'calculate', 'final_answer']


## 8. Eseguiamo l'Agente!

Usiamo `agent.run()` per dare un obiettivo all'agente in linguaggio naturale.

Osserva il **ciclo Think → Act → Observe** nell'output:
- **Thinking**: il modello ragiona sul problema
- **Calling tool**: il modello esegue un'azione
- **Observation**: il risultato del tool viene letto

### Query 1: Fuso orario

In [17]:
# Domanda semplice: richiede un solo tool
risposta = agent.run(
    "Che ore sono adesso a Tokyo e a Los Angeles? Qual è la differenza?"
)

print("\n" + "="*50)
print("RISPOSTA FINALE:")
print(risposta)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Che ore sono adesso a Tokyo e a Los Angeles? Qual è la differenza?                                              │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
I need to find the current time in Tokyo and Los Angeles, then calculate the difference between them.              
                                                                                                                   
Let me break this down into steps:                                                                                 
1. Get the current time in Tokyo                                                                                   
2. Get the current time in Los Angeles                                                                             
3. Calculate the difference between these times                                                                    
                                                                                                                   
I'll use the `get_current_time_in_timezone` tool for this.                                                         
<code>                                                                                                             
tokyo_time = get_current_time_in_timezone("Asia/Tokyo")                                                            
print(f"Current time in Tokyo: {tokyo_time}")                                                                      
                                                                                                                   
la_time = get_current_time_in_timezone("America/Los_Angeles")                                                      
print(f"Current time in Los Angeles: {la_time}")                                                                   
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  tokyo_time = get_current_time_in_timezone("Asia/Tokyo")                                                          
  print(f"Current time in Tokyo: {tokyo_time}")                                                                    
                                                                                                                   
  la_time = get_current_time_in_timezone("America/Los_Angeles")                                                    
  print(f"Current time in Los Angeles: {la_time}")                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Current time in Tokyo: L'ora locale in Asia/Tokyo è: 2026-05-12 18:10:32
Current time in Los Angeles: L'ora locale in America/Los_Angeles è: 2026-05-12 02:10:32

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Based on my analysis, I can now provide the final answer to the question about the current time in Tokyo and Los   
Angeles, and their difference.                                                                                     
                                                                                                                   
Current time in Tokyo: 2026-05-12 18:10:32                                                                         
Current time in Los Angeles: 2026-05-12 02:10:32                                                                   
Difference: 16 hours                                                                                               
                                                                                                                   
Tokyo is 16 hours ahead of Los Angeles.                                                                            
<code>                                                                                                             
final_answer("Current time in Tokyo: 2026-05-12 18:10:32, Current time in Los Angeles: 2026-05-12 02:10:32,        
Difference: 16 hours")                                                                                             
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Current time in Tokyo: 2026-05-12 18:10:32, Current time in Los Angeles: 2026-05-12 02:10:32,      
  Difference: 16 hours")                                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Current time in Tokyo: 2026-05-12 18:10:32, Current time in Los Angeles: 2026-05-12 02:10:32, 
Difference: 16 hours

[Step 3: Duration 2.93 seconds| Input tokens: 7,850 | Output tokens: 543]


RISPOSTA FINALE:
Current time in Tokyo: 2026-05-12 18:10:32, Current time in Los Angeles: 2026-05-12 02:10:32, Difference: 16 hours


### Query 2: Ricerca Web + Calcolo

Questa query richiede che l'agente usi più tool in sequenza: prima cerca su internet, poi eventualmente calcola.

In [18]:
# Domanda che combina ricerca e ragionamento
risposta = agent.run(
    "Cerca quanti parametri ha il modello Qwen2.5-Coder-32B e calcola quanta memoria "
    "servirebbe per caricarlo in float16 (2 byte per parametro). Rispondi in GB."
)

print("\n" + "="*50)
print("RISPOSTA FINALE:")
print(risposta)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Cerca quanti parametri ha il modello Qwen2.5-Coder-32B e calcola quanta memoria servirebbe per caricarlo in     │
│ float16 (2 byte per parametro). Rispondi in GB.                                                                 │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: Per rispondere a questa domanda, ho bisogno di due informazioni principali:                               
                                                                                                                   
1. Il numero totale di parametri del modello Qwen2.5-Coder-32B.                                                    
2. Calcolare la memoria necessaria per caricare il modello in float16 (2 byte per parametro).                      
                                                                                                                   
Prima di tutto, cercherò le informazioni sul numero di parametri del modello Qwen2.5-Coder-32B utilizzando una     
ricerca web.                                                                                                       
<code>                                                                                                             
parametro_ricerca = web_search("Qwen2.5-Coder-32B parameters count")                                               
print(parametro_ricerca)                                                                                           
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  parametro_ricerca = web_search("Qwen2.5-Coder-32B parameters count")                                             
  print(parametro_ricerca)                                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[Qwen2.5-Coder 32B Instruct API | Fireworks AI](https://fireworks.ai/models/fireworks/qwen2p5-coder-32b-instruct)
On-demand deployments give you dedicated GPUs for Qwen2.5-Coder 32B Instruct using ... How many parameters does 
Qwen2.5-Coder 32B Instruct have?

[Qwen2.5-Coder Technical Report](https://arxiv.org/html/2409.12186v3)
Table 1 outlines the architecture of Qwen2.5-Coder across six different model sizes: 0.5B, 1.5B, 3B, 7B, 14B, and 
32B parameters.

[Alibaba Qwen3 coder with 480 billion parameters](https://ai-rockstars.com/alibaba-qwen3-coder/)
Alibaba Qwen3 coder with 480 billion parameters: Open-source AI outperforms GPT-4 ... Qwen3-Coder achieves an 
accuracy of 61.8 percent on SWE-Bench ...

[Qwen2.5-Coder-32B-Instruct vs. Claude 3.5 Sonnet vs. 
GPT-4o:](https://techwavearena.com/qwen2-5-coder-32b-instruct-vs-claude-3-5-sonnet-vs-gpt-4o-coding-llm-comparison/
)
Qwen2.5-Coder-32B-Instruct leads with a massive 32.5 billion parameters , making it one of the most powerful 
open-source models available.

[Qwen2.5-VL-32B: Smarter and Lighter | Hacker News](https://news.ycombinator.com/item?id=43464068)
What s your take on why certain countries will have it cheaper and subsidies being at the forefront? An energy 
driven race to the bottom, is perhaps ...

[llama-bench with Qwen 2.5 Coder 32B and AMD EPYC 
9554](https://ahelpme.com/ai/llamacpp-ai/llama-bench-the-qwen-2-5-coder-32b-and-amd-epyc-9554-cpu/)
root/llama.cpp/build/bin/llama-bench --numa distribute -t 64 -p 0 -n 128,256,512,1024,2048 -m 
/root/models/tests/qwen2.5-coder-32b-instruct-q4_k_m ...

[llama-bench with Qwen2 32B (QwQ-32B) and AMD EPYC 
9554](https://ahelpme.com/ai/llamacpp-ai/llama-bench-the-qwen2-32b-qwq-32b-and-amd-epyc-9554-cpu/)
For the LLM model the Qwen2 32B (QwQ-32B) with different quantization are used to show the difference in token 
generation per second and memory ...

[daVinci-Env: Open SWE Environment Synthesis at Scale](https://arxiv.org/html/2603.13023v1)
... validate OpenSWE’s effectiveness: OpenSWE-32B and OpenSWE-72B achieve 62.4% and 66.0% on SWE-bench Verified, 
establishing SOTA among Qwen2.5 series ...

[Setup AI Code Completions In VS Code Using Ollama - 
Krupesh](https://krupeshanadkat.com/setup-ai-code-completions-in-vs-code-using-ollama/)
I have considered Qwen2.5 Coder with 7B parameters to be suitable for my case. ... You can see list of all 
available parameters for Qwen2.5 Coder ...

[Qwen 2.5 vs GPT-4o: Which model is better for coding? - Bind 
AI](https://blog.getbind.co/qwen-2-5-vs-gpt-4o-comparison/)
Besides the regular Qwen2.5, there ’ s Qwen2.5-coder for coding tasks and Qwen2.5-math for arithmetic and logical 
tasks.

Out: None

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Calcolo della memoria necessaria in byte                                                                       
  parametri = 32 * 10**9  # 32 miliardi di parametri                                                               
  byte_per_parametro = 2  # float16                                                                                
  memoria_totale_byte = parametri * byte_per_parametro                                                             
                                                                                                                   
  # Conversione in GB                                                                                              
  memoria_gb = memoria_totale_byte / (1024**3)                                                                     
                                                                                                                   
  print(f"Memoria richiesta: {memoria_gb:.2f} GB")                                                                 
  final_answer(memoria_gb)                                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Memoria richiesta: 59.60 GB

Final answer: 59.604644775390625

[Step 2: Duration 3.14 seconds| Input tokens: 5,712 | Output tokens: 394]


RISPOSTA FINALE:
59.604644775390625


### Query 3: Test libero

Prova a fare una domanda a tua scelta! L'agente ha a disposizione:
- Ricerca web (può trovare informazioni aggiornate)
- Orologio mondiale
- Calcolatrice

In [ ]:
# Modifica questa query con la tua domanda!
mia_domanda = "Qual è la capitale della Nuova Zelanda e che ore sono lì adesso?"

risposta = agent.run(mia_domanda)

print("\n" + "="*50)
print("RISPOSTA FINALE:")
print(risposta)

## 9. Interfaccia Web con Gradio (Opzionale)

`smolagents` include un'integrazione con **Gradio** per creare una chat UI nel browser con una sola riga di codice.

> **Nota:** questa cella lancia un server locale. Apparirà un link `http://127.0.0.1:7860`. Fermalo con il pulsante `■ Stop` nella toolbar del notebook o con `Kernel → Interrupt`.

In [19]:
from smolagents import GradioUI

# Lancia l'interfaccia chat nel browser
# Interrompi l'esecuzione della cella per fermare il server
GradioUI(agent).launch()

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://7e2f5f5ded163245c1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ dimmi che tempo fa oggi a Trento, (italia)?                                                                     │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: Per conoscere il tempo a Trento in Italia, utilizzerò lo strumento `get_current_time_in_timezone`         
specificando il fuso orario italiano (Europe/Rome).                                                                
<code>                                                                                                             
tempo_trento = get_current_time_in_timezone("Europe/Rome")                                                         
print(tempo_trento)                                                                                                
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  tempo_trento = get_current_time_in_timezone("Europe/Rome")                                                       
  print(tempo_trento)                                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Dalla ricerca, ho trovato informazioni sul meteo a Trento. Secondo i risultati, oggi a Trento ci saranno nuvolosità
sparsa con temperature che variano tra i 14°C e i 19°C. Ci saranno piogge sparse al mattino e al pomeriggio, con   
nuvolosità leggera alla sera.                                                                                      
                                                                                                                   
Fornisco questa informazione come risposta finale.                                                                 
<code>                                                                                                             
final_answer("A Trento oggi ci saranno nuvolosità sparsa con temperature che variano tra i 14°C e i 19°C. Ci       
saranno piogge sparse al mattino e al pomeriggio, con nuvolosità leggera alla sera.")                              
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("A Trento oggi ci saranno nuvolosità sparsa con temperature che variano tra i 14°C e i 19°C. Ci     
  saranno piogge sparse al mattino e al pomeriggio, con nuvolosità leggera alla sera.")                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: A Trento oggi ci saranno nuvolosità sparsa con temperature che variano tra i 14°C e i 19°C. Ci 
saranno piogge sparse al mattino e al pomeriggio, con nuvolosità leggera alla sera.

[Step 6: Duration 5.13 seconds| Input tokens: 23,353 | Output tokens: 745]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7e2f5f5ded163245c1.gradio.live


## 10. Aggiungere un Nuovo Tool — Esercizio

Ora tocca a te! Completa il tool qui sotto e aggiungilo all'agente.

Idee:
- Un tool che converte valute (es. EUR → USD)
- Un tool che conta le parole in un testo
- Un tool che genera una battuta casuale

In [ ]:
@tool
def my_tool(input_text: str) -> str:  # Modifica i tipi se necessario
    """Descrivi cosa fa il tuo tool.
    Args:
        input_text: Descrivi questo argomento.
    """
    # Scrivi qui la tua logica
    return f"Il tuo tool ha ricevuto: {input_text}"


# Ricrea l'agente con il nuovo tool
agent_con_nuovo_tool = CodeAgent(
    model=model,
    tools=[search_tool, get_current_time_in_timezone, calculate, my_tool],
    max_steps=5,
    verbosity_level=1,
)

print(f"Tool disponibili: {[t.name for t in agent_con_nuovo_tool.tools.values()]}")

## Riepilogo

In questo notebook abbiamo:

| Passo | Cosa abbiamo fatto |
|-------|-------------------|
| Tool custom | Creato tool con `@tool`, type hints e docstring |
| Tool built-in | Usato `DuckDuckGoSearchTool` di smolagents |
| Modello | Connesso a `Qwen2.5-Coder-32B` via API HuggingFace |
| Agente | Costruito un `CodeAgent` con più tool |
| Esecuzione | Osservato il ciclo Think → Act → Observe |
| UI | Lanciato un'interfaccia chat con Gradio |